# 数据接入核验

## tl;dr
已有 372 个共同文件日期和 210 条名称候选；三类港股期货小样本可读。文件存在不等于成分行情完整，尚未执行回测。

## Context & Methods
伴随审计，读取 scripts/inventory_remote.py 已生成的证据，不重新扫描远端。

### Key Assumptions
日期集合仅表达文件存在；抽查结果不能推广至全体基金日。PCF 语义、单位和时间标签仍待核准。

In [1]:
from pathlib import Path
import json
root = Path.cwd()
if not (root / 'data/inventory').exists(): root = root.parent
p = json.loads((root / 'data/inventory/remote_inventory.json').read_text())
tws = json.loads((root / 'data/inventory/tws_probe.json').read_text())
print('Evidence time:', p['generated_at'])

Evidence time: 2026-09-06T06:10:41.405571+00:00


## Data
文件清单的日期、计数与可重复性。

In [2]:
sets = {k: {x['date'] for x in v} for k,v in p['files'].items()}
for k,v in sets.items(): print(k, len(v), min(v), max(v))
common = sorted(set.intersection(*sets.values()))
assert common == p['common_dates']
print('Common file dates:',len(common))
print('Candidate rows:',len(p['candidates']))
print('Sampled dates:',len(p['samples']))

cn_etf_1m 5212 20050223 20260803
hk_trades 402 20250102 20260825
pcf_detail 5210 20050223 20260813
Common file dates: 372
Candidate rows: 210
Sampled dates: 14


## Results
520600 的替代标志变化和成分成员缺失；数量是证券行数，不是风险权重。

In [3]:
for sample in p['samples']:
    detail = sample['pcf']['520600']
    coverage = sample['hk'].get('pilot_520600_component_members',{})
    assert detail['rows'] == detail['distinct_components']
    assert detail['rows'] == int(detail['header_component_count'])
    print(sample['date'], detail['rows'],detail['flags'], 'missing members:',coverage.get('unmatched'))

20250102 40 {'退补': 40} missing members: []
20250304 40 {'退补': 40} missing members: []
20250429 38 {'退补': 38} missing members: [{'code': '02362', 'name': '金川国际'}]
20250625 42 {'退补': 42} missing members: [{'code': '02362', 'name': '金川国际'}]
20250818 41 {'退补': 41} missing members: [{'code': '00489', 'name': '东风集团'}]
20251016 38 {'退补': 38} missing members: []
20251209 38 {'允许': 38} missing members: []
20260204 46 {'允许': 46} missing members: [{'code': '09696', 'name': '天齐锂业'}]
20260409 43 {'允许': 43} missing members: [{'code': '02402', 'name': '亿华通'}]
20260608 43 {'允许': 43} missing members: []
20260730 50 {'允许': 50} missing members: []
20260731 50 {'允许': 50} missing members: []
20260803 50 {'允许': 50} missing members: []
20260813 50 {'未知': 50} missing members: []


### TWS 实际返回
只是一天的历史权限与合约发现验证，不是全历史下载验收。

In [4]:
assert tws['api_ready']
for result in tws['results']:
    assert len(result['contracts']) == 1
    assert result['bar_count'] > 0
    contract = result['contracts'][0]
    print(result['symbol'],contract['symbol'],contract['localSymbol'],result['bar_count'])

HSI HSI HSIN6 345
HHI HHI.HK HHIN6 345
HTI HSTECH HTIN6 345


## Takeaways
先核对 PCF、港股停复牌/公司行动、分钟时间与量单位、汇率，再按 config/pilot.json 试跑。基金名称分类需发行人文件验证。详细接入流程见 docs/STANDARD_WORKFLOW.md。